# Сравнение минимальной длины 250 и 200 при fastp-фильтрации лёгких цепей мыши

Сравниваются объединённые pRESTO-последовательности после exact collapse, одинаковая IgBLAST-аннотация с базой BALB/cByJ, строгий набор продуктивных полных IGK/IGL-последовательностей от V до J и остаточные праймерные мотивы. Ветка min250 не перезаписывается.


In [ ]:
from pathlib import Path
import csv,gzip,json,os,statistics,subprocess,sys,time
from collections import Counter
REPO=Path.cwd().resolve()
if not (REPO/'scripts/mixed_chain_truth.py').exists():REPO=REPO.parent
sys.path.insert(0,str(REPO))
from scripts.mixed_chain_truth import build_airr_truth
ENV=Path('/Users/epishkin/mamba/envs/bcr_env');BASE=REPO/'results/PRJNA1226555';GERMLINE=BASE/'references/germline'
ROOTS={'min250':BASE/'branches/fastp_q30_u40_min250','min200':BASE/'branches/fastp_q30_u40_min200'}
for p in [ENV/'bin/CollapseSeq.py',ENV/'bin/igblastn',GERMLINE/'igblast_db/balbcbyj_igkv_iglv.nsq',GERMLINE/'igblast_db/all_strains_IGKLJ.nsq']:assert p.exists(),p
def run_visible(cmd,out,err,env=None):
 out=Path(out);err=Path(err);out.parent.mkdir(parents=True,exist_ok=True);t=time.monotonic()
 with out.open('w') as oh,err.open('w') as eh:
  p=subprocess.Popen([str(x) for x in cmd],cwd=REPO,stdout=oh,stderr=eh,text=True,env=env);print('PID',p.pid,flush=True)
  while p.poll() is None:print(f'PID={p.pid} elapsed={(time.monotonic()-t)/60:.1f}min',flush=True);time.sleep(30)
 if p.returncode:raise RuntimeError(f'rc={p.returncode}; inspect {err}')
def fasta_count(p):
 with Path(p).open() as h:return sum(x.startswith('>') for x in h)
def tsv_count(p):
 with Path(p).open(errors='replace') as h:return max(0,sum(1 for _ in h)-1)
def fasta_ids(path):
 ids=[]
 with Path(path).open() as h:
  for line in h:
   if line.startswith('>'):ids.append(line[1:].strip().split()[0])
 return ids
def airr_ids(path):
 with Path(path).open(errors='replace') as h:return [r['sequence_id'] for r in csv.DictReader(h,delimiter='\t')]
def assert_airr_query_identity(fasta,airr):
 expected=fasta_ids(fasta);observed=airr_ids(airr)
 if len(expected)!=len(set(expected)):raise AssertionError(f'duplicate FASTA IDs: {len(expected)-len(set(expected))}')
 if len(observed)!=len(set(observed)):raise AssertionError(f'duplicate AIRR sequence_ids: {len(observed)-len(set(observed))}')
 missing=set(expected)-set(observed);extra=set(observed)-set(expected)
 assert not missing and not extra,(f'AIRR query-ID mismatch: missing={len(missing)} extra={len(extra)}',sorted(missing)[:5],sorted(extra)[:5])
 assert len(observed)==len(expected),(len(observed),len(expected))
 return len(expected)
def airr_query_identity_ok(fasta,airr):
 try:assert_airr_query_identity(fasta,airr);return True
 except (AssertionError,KeyError,ValueError,OSError) as exc:print(f'[rerun] invalid AIRR {airr}: {exc}',flush=True);return False
print({k:str(v) for k,v in ROOTS.items()})


In [ ]:
# Одинаково преобразовать объединённые pRESTO FASTQ и последовательности после exact collapse.
for label,root in ROOTS.items():
 merged=root/'merged/SRR32426580_all_assemble-pass.fastq.gz';work=root/'annotation';work.mkdir(parents=True,exist_ok=True)
 fasta=work/'SRR32426580_all_assemble-pass.fasta';collapsed=work/'SRR32426580.collapse-unique.fasta';assert merged.exists(),merged
 if not fasta.exists():
  n=0
  with gzip.open(merged,'rt') as ih,fasta.open('w') as oh:
   while True:
    head=ih.readline()
    if not head:break
    seq=ih.readline().strip();ih.readline();ih.readline();oh.write('>'+head[1:].split()[0]+'\n'+seq+'\n');n+=1
  print(label,'FASTA',n)
 if not collapsed.exists():run_visible([ENV/'bin/CollapseSeq.py','-s',fasta,'-o',collapsed,'--fasta','-n','0'],work/'collapse.stdout.log',work/'collapse.stderr.log')
 print(label,'assembled',fasta_count(fasta),'collapsed',fasta_count(collapsed))


In [ ]:
# Одинаковая IgBLAST-аннотация: V из BALB/cByJ и light-J для всех линий.
for label,root in ROOTS.items():
 work=root/'annotation';collapsed=work/'SRR32426580.collapse-unique.fasta';airr=work/'SRR32426580_balbcbyj_igkl.airr.tsv';tmp=work/'SRR32426580_balbcbyj_igkl.airr.tsv.rerun.tmp';expected=fasta_count(collapsed)
 if not (airr.exists() and airr_query_identity_ok(collapsed,airr)):
  tmp.unlink(missing_ok=True);env=os.environ.copy();env['IGDATA']=str(GERMLINE/'igdata_balbcbyj')
  cmd=[ENV/'bin/igblastn','-query',collapsed,'-organism','balbcbyj','-ig_seqtype','Ig','-germline_db_V',GERMLINE/'igblast_db/balbcbyj_igkv_iglv','-germline_db_D',GERMLINE/'ncbi_mouse/mouse_gl_D','-germline_db_J',GERMLINE/'igblast_db/all_strains_IGKLJ','-auxiliary_data',GERMLINE/'ogrdb_balbcbyj/all_strains_IGKLJ.aux','-domain_system','imgt','-outfmt','19','-num_threads','8','-out',tmp]
  run_visible(cmd,work/'igblast.stdout.log',work/'igblast.stderr.log',env);assert tsv_count(tmp)==expected,(label,tsv_count(tmp),expected);assert_airr_query_identity(collapsed,tmp);tmp.replace(airr)
 verified=assert_airr_query_identity(collapsed,airr)
 print(label,'AIRR rows/query IDs verified',verified)


In [ ]:
# Исходный AIRR, результативность предыдущих стадий и строгие сводки продуктивных полных V–J.
def yes(v):return str(v).lower() in {'true','t','1','yes'}
def median_or_none(values):return statistics.median(values) if values else None
report={}
for label,root in ROOTS.items():
 work=root/'annotation';airr=work/'SRR32426580_balbcbyj_igkl.airr.tsv';collapsed=work/'SRR32426580.collapse-unique.fasta';raw=Counter();loci=Counter();merged_lengths=[];vj_lengths=[]
 verified_queries=assert_airr_query_identity(collapsed,airr)
 with airr.open(errors='replace') as h:
  for r in csv.DictReader(h,delimiter='\t'):
   raw['rows']+=1;raw['productive']+=yes(r.get('productive'));raw['complete_vdj']+=yes(r.get('complete_vdj'));raw['productive_complete_no_stop']+=yes(r.get('productive')) and yes(r.get('complete_vdj')) and not yes(r.get('stop_codon'));loci[r.get('locus','UNASSIGNED')]+=1
   sequence=r.get('sequence') or ''
   if sequence:merged_lengths.append(len(sequence))
   try:vj_lengths.append(int(r['j_sequence_end'])-int(r['v_sequence_start'])+1)
   except (ValueError,KeyError,TypeError):pass
 truth=build_airr_truth(input_tsv=airr,output_dir=work/'strict_truth',run=f'SRR32426580_{label}',source=f'PRJNA1226555_{label}',expected_loci={'IGK','IGL'})
 manifest=work/'strict_truth'/f'SRR32426580_{label}_template_manifest.tsv';strict_loci=Counter()
 with manifest.open() as h:
  for r in csv.DictReader(h,delimiter='\t'):strict_loci[r['locus']]+=1
 trim_summary=json.loads((root/'trimmed/trim_summary.json').read_text())
 assembly_summary=json.loads((root/'merged/assembly_summary.json').read_text())
 qc={}
 for stage,view in [('trimmed','primary'),('pr_trimmed','primary'),('merged','primary'),('merged','branch_diagnostics')]:
  suffix='' if view=='primary' else '_branch_diagnostics';manifest_path=root/stage/f'multiqc{suffix}/qc_run_manifest.json'
  qc[f'{stage}_{view}']={'manifest':str(manifest_path),'available':manifest_path.is_file()}
 report[label]={'upstream':{'trim_summary':trim_summary,'assembly_summary':assembly_summary},'collapsed_queries':fasta_count(collapsed),'annotation_query_identity_verified':verified_queries,'raw_airr':dict(raw),'airr_loci':dict(loci),'merged_length_median':median_or_none(merged_lengths),'vj_length_median':median_or_none(vj_lengths),'strict':truth,'strict_unique_loci':dict(strict_loci),'qc':qc}
print(json.dumps(report,indent=2))


In [ ]:
# Остаточные известные IGL-мотивы и кандидаты Cκ, включая фланки за пределами V/J.
def rc(s):return s.translate(str.maketrans('ACGT','TGCA'))[::-1]
igl=['AGCTCTTCAGAGGAAGGTGG','AGCTCTTCAGGGGAAGGTGG','AGCTCCTCAGAGGAAGGTGG','AGCTCCTCAGGGGAAGGTGG'];motifs={**{f'IGL{i+1}':x for i,x in enumerate(igl)},'CK_candidate':'GATGGTGGGAAGATGGATAC'};audit={}
for label,root in ROOTS.items():
 airr=root/'annotation/SRR32426580_balbcbyj_igkl.airr.tsv';counts=Counter();den=Counter()
 with airr.open(errors='replace') as h:
  for r in csv.DictReader(h,delimiter='\t'):
   locus=r.get('locus') or 'UNASSIGNED';seq=r.get('sequence','').upper();den[locus]+=1
   try:s=int(r['v_sequence_start'])-1;e=int(r['j_sequence_end'])
   except (ValueError,KeyError):s=0;e=len(seq)
   flank=seq[:s]+seq[e:]
   for name,m in motifs.items():
    if m in seq or rc(m) in seq:counts[(locus,name,'full')]+=1
    if m in flank or rc(m) in flank:counts[(locus,name,'outside_VJ')]+=1
 audit[label]={'denominators':dict(den),'counts':{'|'.join(k):v for k,v in sorted(counts.items())}}
comparison_out=BASE/'comparisons/light_fastp_minlen_comparison.json';comparison_tmp=Path(str(comparison_out)+'.tmp');comparison_tmp.write_text(json.dumps({'comparison':report,'residual_motifs':audit},indent=2)+'\n');comparison_tmp.replace(comparison_out);print(json.dumps(audit,indent=2));print(comparison_out)
